# Carregamento e inspeção inicial do TrashNet

Inspeção inicial das divisões, atributos, rótulos, distribuição das classes e amostras visuais. Este notebook não treina nem avalia um modelo.

> **Armazenamento:** a página do Hugging Face informa tamanho total de cerca de 3,68 GB. A execução de `load_dataset` pode transferir arquivos para o cache do ambiente. Verifique o espaço disponível antes da execução.

In [ ]:
%pip -q install datasets matplotlib

## 1. Carregar do Hugging Face

`load_dataset` busca o conjunto pelo identificador `garythung/trashnet` e o deixa disponível como objeto Python. Não precisamos baixar e extrair o ZIP manualmente para este fluxo.

In [ ]:
from pathlib import Pathfrom datasets import load_datasetcaminhos_candidatos = [    Path("trashnet/dataset-resized/dataset-resized"),    Path("../trashnet/dataset-resized/dataset-resized"),]caminho_dataset = next((caminho for caminho in caminhos_candidatos if caminho.is_dir()), None)if caminho_dataset is None:    caminhos_verificados = ", ".join(str(caminho.resolve()) for caminho in caminhos_candidatos)    raise FileNotFoundError(f"Pasta do TrashNet não encontrada. Caminhos verificados: {caminhos_verificados}")ds = load_dataset("imagefolder", data_dir=str(caminho_dataset))ds

## 2. Conferir divisões, campos e classes

A página do dataset apresenta uma divisão chamada `train`. Essa divisão não substitui conjuntos separados de treino, validação e teste; a separação será definida posteriormente.

In [ ]:
print("Divisões disponíveis:", list(ds.keys()))
treino = ds["train"]
print("Quantidade de exemplos:", len(treino))
print("Campos:", treino.features)
print("Nomes das classes:", treino.features["label"].names)

## 3. Contar exemplos por classe

A contagem descreve o dataset carregado e não é uma métrica do modelo. Ela permite identificar diferenças no número de exemplos entre as classes.

In [ ]:
from collections import Counter

contagens = Counter(treino["label"])
nomes = treino.features["label"].names
for indice, nome in enumerate(nomes):
    print(f"{nome:10} {contagens[indice]:5} imagens")

## 4. Olhar exemplos

A inspeção visual permite verificar se as imagens são legíveis e se os rótulos parecem coerentes. As imagens do TrashNet foram fotografadas principalmente sobre fundo branco; os resultados serão comparados posteriormente com fotos reais da equipe.

In [ ]:
import matplotlib.pyplot as plt

quantidade = min(6, len(treino))
fig, eixos = plt.subplots(1, quantidade, figsize=(3 * quantidade, 3))
if quantidade == 1:
    eixos = [eixos]
for indice, eixo in enumerate(eixos):
    exemplo = treino[indice]
    eixo.imshow(exemplo["image"].convert("RGB"))
    eixo.set_title(nomes[exemplo["label"]])
    eixo.axis("off")
plt.tight_layout()

## Saídas a registrar

1. Divisões disponíveis.
2. Número total de exemplos carregados.
3. Nomes e contagens por classe.
4. Coerência visual entre imagens e rótulos.

A próxima etapa é criar uma divisão estratificada de treino, validação e teste. As fotos reais serão mantidas separadas para permitir uma avaliação independente do domínio de captura.